In [1]:
#问题2：理论分析，如果将测试集合中所有样本都预测为最主要类别，分类精度Accuracy是多少？查全率、查准率、F1得分分别是多少？

In [2]:
# 公共初始化代码｜输出：环境、数据来源、固定划分与输出目录
from pathlib import Path
from datetime import datetime
import hashlib
import json
import sys
import uuid
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

PROJECT_DIR = Path(r"D:\Python for Data Analysis\notebooks\imbalanced_classification")
SOURCE_DIR = PROJECT_DIR / "runs/20261020_221259_7f297c"
assert Path(sys.executable).resolve() == Path(r"D:\anaconda\envs\pda312\python.exe").resolve(), "请选择pda312内核"
source_manifest = json.loads((SOURCE_DIR / "results/manifest.json").read_text(encoding="utf-8"))
with np.load(SOURCE_DIR / "data/dataset_and_split.npz") as saved:
    X, y = saved["X"], saved["y"]
    train_idx, test_idx = saved["train_idx"], saved["test_idx"]
data_hash = hashlib.sha256(X.tobytes() + y.tobytes() + train_idx.tobytes() + test_idx.tobytes()).hexdigest()
assert data_hash == source_manifest["data_split_sha256"], "数据或划分与第一题记录不一致"
LABELS = np.arange(10)
y_train, y_test = y[train_idx], y[test_idx]
assert np.array_equal(np.unique(y_train), LABELS) and np.array_equal(np.unique(y_test), LABELS)
Q2_DIR = PROJECT_DIR / "q2_runs" / (datetime.now().strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:6])
Q2_DIR.mkdir(parents=True, exist_ok=False)
print("问题2 v1.0；Python", sys.version.split()[0], "；scikit-learn", sklearn.__version__)
print("第一题数据来源：", SOURCE_DIR)
print("训练/测试样本数：", len(y_train), len(y_test))
print("本题输出目录：", Q2_DIR)

问题2 v1.0；Python 3.12.13 ；scikit-learn 1.9.0
第一题数据来源： D:\Python for Data Analysis\notebooks\imbalanced_classification\runs\20261020_221259_7f297c
训练/测试样本数： 3600 1200
本题输出目录： D:\Python for Data Analysis\notebooks\imbalanced_classification\q2_runs\20261020_222308_143f6b


In [3]:
# ；输出：常数预测类别及逐类指标
train_counts = np.bincount(y_train, minlength=10)
# 仅根据训练集选择常数类别；flatnonzero升序返回标签，明确处理并列。
constant_class = int(np.flatnonzero(train_counts == train_counts.max())[0])
y_pred = np.full_like(y_test, constant_class)
precision, recall, f1, support = precision_recall_fscore_support(
    y_test, y_pred, labels=LABELS, average=None, zero_division=0
)
per_class = pd.DataFrame({
    "class_label": LABELS, "support": support,
    "precision": precision, "recall": recall, "f1": f1,
})
accuracy = accuracy_score(y_test, y_pred)
print("训练集确定的常数预测类别：", constant_class)
print("测试集全部预测为该类，测试集真实计数如下：")
display(per_class.round(6))
print(f"Accuracy = {accuracy:.6f}（{accuracy:.4%}）")

训练集确定的常数预测类别： 0
测试集全部预测为该类，测试集真实计数如下：


,class_label,support,precision,recall,f1
0,0,379,0.315833,1.0,0.480051
1,1,225,0.000000,0.0,0.000000
2,2,155,0.000000,0.0,0.000000
3,3,114,0.000000,0.0,0.000000
4,4,91,0.000000,0.0,0.000000
5,5,74,0.000000,0.0,0.000000
6,6,56,0.000000,0.0,0.000000
7,7,43,0.000000,0.0,0.000000
8,8,37,0.000000,0.0,0.000000
9,9,26,0.000000,0.0,0.000000


Accuracy = 0.315833（31.5833%）


In [4]:
# 依赖：常数预测；输出：macro/micro/weighted
average_rows = []
for average in ("macro", "micro", "weighted"):
    p_score, r_score, f_score, _ = precision_recall_fscore_support(
        y_test, y_pred, labels=LABELS, average=average, zero_division=0
    )
    average_rows.append({"average": average, "precision": p_score,
                         "recall": r_score, "f1": f_score})
averages = pd.DataFrame(average_rows)
display(averages.round(6))
print("口径：全部10类、zero_division=0；Accuracy单独报告，不当成Precision。")

,average,precision,recall,f1
0,macro,0.031583,0.100000,0.048005
1,micro,0.315833,0.315833,0.315833
2,weighted,0.099751,0.315833,0.151616


口径：全部10类、zero_division=0；Accuracy单独报告，不当成Precision。


In [5]:
# 辅助代码：理论值与sklearn输出交叉核对｜输出：代入数值及核验状态
N, K = len(y_test), len(LABELS)
n_c = int(np.count_nonzero(y_test == constant_class))
p = n_c / N
theory_averages = pd.DataFrame([
    {"average": "macro", "precision": p/K, "recall": 1/K, "f1": 2*p/(K*(1+p))},
    {"average": "micro", "precision": p, "recall": p, "f1": p},
    {"average": "weighted", "precision": p**2, "recall": p, "f1": 2*p**2/(1+p)},
])
theory_per_class = np.zeros((K, 3))
theory_per_class[constant_class] = [p, 1.0, 2*p/(1+p)]
np.testing.assert_allclose(per_class[["precision", "recall", "f1"]], theory_per_class, atol=1e-12)
np.testing.assert_allclose(averages[["precision", "recall", "f1"]],
                           theory_averages[["precision", "recall", "f1"]], atol=1e-12)
assert np.isclose(accuracy, p)
print(f"代入：K={K}, c={constant_class}, N={N}, n_c={n_c}, p={p:.9f}")
print("公式与sklearn逐类指标、三种平均指标及Accuracy一致。")

代入：K=10, c=0, N=1200, n_c=379, p=0.315833333
公式与sklearn逐类指标、三种平均指标及Accuracy一致。


In [6]:
# 辅助代码：保存真实运行结果｜输出：CSV和来源记录；将此目录发给助手绘图
per_class.to_csv(Q2_DIR / "q2_per_class.csv", index=False, encoding="utf-8-sig")
averages.to_csv(Q2_DIR / "q2_averages.csv", index=False, encoding="utf-8-sig")
pd.DataFrame({"sample_id": test_idx, "y_true": y_test, "y_pred": y_pred}).to_csv(
    Q2_DIR / "q2_predictions.csv", index=False, encoding="utf-8-sig"
)
record = dict(code_version="1.0", source_dir=str(SOURCE_DIR), data_split_sha256=data_hash,
              python=sys.version, sklearn=sklearn.__version__, executable=sys.executable,
              created_at=datetime.now().astimezone().isoformat(), constant_class=constant_class,
              N=N, n_c=n_c, accuracy=float(accuracy), labels=LABELS.tolist(), zero_division=0,
              formula_check="passed", output_dir=str(Q2_DIR))
(Q2_DIR / "q2_manifest.json").write_text(json.dumps(record, ensure_ascii=False, indent=2), encoding="utf-8")
print("保存完成：", Q2_DIR)
print("请保留两段核心代码与输出截图，以及公式核验和保存完成输出。")

保存完成： D:\Python for Data Analysis\notebooks\imbalanced_classification\q2_runs\20261020_222308_143f6b
请保留两段核心代码与输出截图，以及公式核验和保存完成输出。
